# Policy Rate - Data Cleaning 

### Data Info:
* **Indicator:** `Облікова ставка (Policy Rate)`
* **Source:** National Bank of Ukraine: https://bank.gov.ua/ua/monetary/archive-rish
* **Raw File:** `data_raw/Policy_Rate.xlsx`
* **Output:** `data_processed/Policy_Rate_clean.xlsx`  

### Cleaning Notes:
- data is event-based, not monthly. Each row represents a decision date from which a new policy rate became effective. Therefore, monthly observations were constructed by forward-filling the most recent available rate until the next decision date.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
file_path = Path("../../data_raw/Policy_Rate.xlsx")
output_path = Path("../../data_processed/Policy_Rate_cleaned.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

sheet_name = "Sheet1"
df = pd.read_excel(file_path, sheet_name=sheet_name, header=0)
df

,year,from date,value
0,2025,12.12,"15,5"
1,2025,24.10,"15,5"
2,2025,12.09,"15,5"
3,2025,25.07,"15,5"
4,2025,6.06,"15,5"
...,...,...,...
76,2016,24.06,16.5
77,2016,27.05,18
78,2016,22.04,19
79,2016,4.03,22


In [3]:
# clean 

df["year"] = pd.to_numeric(df["year"], errors="coerce")

df["value"] = (
    df["value"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# build date 
# Note: Excel stores dd.mm as a float (so 28.10 -> 28.1) 
# Therefore, as a month we take fractional part × 100 

_fd = pd.to_numeric(df["from date"], errors="coerce")
_day   = _fd.apply(lambda x: int(round(x))          if pd.notna(x) else pd.NA)
_month = _fd.apply(lambda x: int(round((x % 1) * 100)) if pd.notna(x) else pd.NA)

df["decision_date"] = pd.to_datetime(
    df["year"].astype("Int64").astype(str)
    + "-" + _month.astype("Int64").astype(str).str.zfill(2)
    + "-" + _day.astype("Int64").astype(str).str.zfill(2),
    format="%Y-%m-%d",
    errors="coerce"
)

df = df.dropna(subset=["decision_date", "value"]).copy()
df = df.sort_values("decision_date").reset_index(drop=True)
df

,year,from date,value,decision_date
0,2016,29.01,22.0,2016-01-29
1,2016,4.03,22.0,2016-03-04
2,2016,22.04,19.0,2016-04-22
3,2016,27.05,18.0,2016-05-27
4,2016,24.06,16.5,2016-06-24
...,...,...,...,...
76,2025,6.06,15.5,2025-06-06
77,2025,25.07,15.5,2025-07-25
78,2025,12.09,15.5,2025-09-12
79,2025,24.10,15.5,2025-10-24


In [4]:
# transform to monthly end-of-month series

month_ends = pd.date_range(
    start=df["decision_date"].min().to_period("M").to_timestamp("M"),
    end=df["decision_date"].max().to_period("M").to_timestamp("M"),
    freq="M"
)

df_monthly = pd.merge_asof(
    pd.DataFrame({"date": month_ends}),
    df[["decision_date", "value"]].sort_values("decision_date"),
    left_on="date",
    right_on="decision_date",
    direction="backward"
)

df_monthly = df_monthly.drop(columns="decision_date")
df_monthly["indicator"] = "policy_rate"

df_monthly

,date,value,indicator
0,2016-01-31,22.0,policy_rate
1,2016-02-29,22.0,policy_rate
2,2016-03-31,22.0,policy_rate
3,2016-04-30,19.0,policy_rate
4,2016-05-31,18.0,policy_rate
...,...,...,...
115,2025-08-31,15.5,policy_rate
116,2025-09-30,15.5,policy_rate
117,2025-10-31,15.5,policy_rate
118,2025-11-30,15.5,policy_rate


In [5]:
df_final = df_monthly[["date", "value", "indicator"]].copy()
df_final["date"] = df_final["date"].dt.strftime("%Y-%m")

df_final.head(12)

,date,value,indicator
0,2016-01,22.0,policy_rate
1,2016-02,22.0,policy_rate
2,2016-03,22.0,policy_rate
3,2016-04,19.0,policy_rate
4,2016-05,18.0,policy_rate
5,2016-06,16.5,policy_rate
6,2016-07,15.5,policy_rate
7,2016-08,15.5,policy_rate
8,2016-09,15.0,policy_rate
9,2016-10,14.0,policy_rate


In [6]:
# save

df_final.to_excel(output_path, index=False)
print(f"saved to: {output_path}")

saved to: ..\..\data_processed\Policy_Rate_cleaned.xlsx


In [7]:
# check the slippery moments :) 

'''
expected = {
    "2016-11": 14.0, "2017-01": 14.0, "2017-02": 14.0, "2017-11": 13.5,
    "2018-01": 16.0, "2018-02": 16.0, "2019-01": 18.0, "2019-11": 15.5,
    "2021-01": 6.0,  "2021-02": 6.0,  "2023-01": 25.0, "2023-02": 25.0,
    "2023-11": 16.0, "2025-01": 14.5, "2025-02": 14.5,
}

check = df_final.set_index("date")["value"]
errors = {m: (check[m], exp) for m, exp in expected.items() if check.get(m) != exp}

if errors:
    print("Errors:")
    for m, (got, exp) in errors.items():
        print(f"  {m}: got {got}, expected {exp}")
else:
    print("All correct!") 
'''

pass